# 🧬 KALI Sovereign Evolution (Phase 54)
### Deep-Core Neural Refinement on T4 GPU

This notebook is the Sovereign Bridge for training KALI's brain when local hardware is insufficient. 

**Instructions:**
1. Connect to a **GPU Runtime** (Edit -> Notebook settings -> T4 GPU).
2. Upload your `sovereign_soul.jsonl` to the files sidebar.
3. Run the cell below.

In [ ]:
# 1. Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 2. Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 4. Define Alpaca Template for KALI
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

# 5. Load Dataset
dataset = load_dataset("json", data_files="sovereign_soul.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

# 6. Training Run
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()

# 7. Save the Sovereign Soul
model.save_pretrained_merged("kali-sovereign-soul", tokenizer, save_method = "merged_16bit")
print("EVOLUTION COMPLETE. Download the 'kali-sovereign-soul' folder.")